# Fine-tuning LoRA - Modele medical (Phi-3.5 Instruct)

Notebook **experimental** pour fine-tuner `microsoft/Phi-3.5-mini-instruct` sur le dataset
medical prepare par l'equipe DATA (`medical_train.json` / `medical_val.json`).

> Modele NON destine a la production medicale - POC uniquement.

## Avant de commencer
1. **Menu Execution -> Modifier le type d'execution -> GPU (T4)** puis Enregistrer.
2. Executer les cellules **dans l'ordre** (bouton triangle a gauche de chaque cellule).
3. A la cellule d'upload, deposer `medical_train.json` et `medical_val.json`
   (dossier `rendu/DATA/data_clean/medical_data/`).

Livrables attendus (consignes IA) : **lien Colab + metriques (train loss, val loss, epochs)**.


## 1. Verifier le GPU


In [ ]:
!nvidia-smi


## 2. Installer les dependances


In [ ]:
!pip -q install -U transformers==4.44.2 peft==0.13.2 accelerate==0.34.2 \
    bitsandbytes==0.43.3 datasets==2.21.0
print('Dependances installees. Si demande, redemarrer l execution (Runtime > Restart).')


## 3. Charger les donnees
Deposer `medical_train.json` et `medical_val.json` quand la fenetre d'upload apparait.


In [ ]:
from google.colab import files
import json, os

if not (os.path.exists('medical_train.json') and os.path.exists('medical_val.json')):
    print('Depose medical_train.json puis medical_val.json :')
    up = files.upload()

with open('medical_train.json', encoding='utf-8') as f:
    train_raw = json.load(f)
with open('medical_val.json', encoding='utf-8') as f:
    val_raw = json.load(f)

print('train:', len(train_raw), '| val:', len(val_raw))
print('exemple:', train_raw[0])


## 4. Charger le modele Phi-3.5 en 4-bit (QLoRA)
Quantization 4-bit pour tenir sur le GPU gratuit T4 (~15 Go).


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

MODEL_ID = 'microsoft/Phi-3.5-mini-instruct'

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb, device_map='auto', trust_remote_code=True
)
model = prepare_model_for_kbit_training(model)
print('Modele charge :', MODEL_ID)


## 5. Configurer LoRA


In [ ]:
lora = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
    target_modules=['qkv_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()


## 6. Formater et tokeniser
Chaque exemple est mis au format chat de Phi-3.5 : `<|user|> ... <|assistant|> ...`.


In [ ]:
from datasets import Dataset

MAX_LEN = 512

def to_text(ex):
    return {'text': f"<|user|>\n{ex['instruction']}<|end|>\n<|assistant|>\n{ex['output']}<|end|>"}

train_ds = Dataset.from_list(train_raw).map(to_text)
val_ds = Dataset.from_list(val_raw).map(to_text)

def tok(batch):
    out = tokenizer(batch['text'], truncation=True, padding='max_length', max_length=MAX_LEN)
    out['labels'] = out['input_ids'].copy()
    return out

train_tok = train_ds.map(tok, batched=True, remove_columns=train_ds.column_names)
val_tok = val_ds.map(tok, batched=True, remove_columns=val_ds.column_names)
print('Tokenisation terminee.')


## 7. Entrainement
Les metriques (train loss + **eval/val loss**) sont loggees regulierement.


In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

args = TrainingArguments(
    output_dir='phi35_medical_lora',
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=50,
    logging_steps=25,
    eval_strategy='steps',
    eval_steps=100,
    save_strategy='epoch',
    save_total_limit=1,
    fp16=True,
    report_to='none',
    remove_unused_columns=False,
)

collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=collator,
)

trainer.train()


## 8. Metriques d'entrainement (a partager)
Recupere la train loss et la val loss - a coller dans le rendu.


In [ ]:
import json
hist = trainer.state.log_history

train_pts = [(h['step'], h['loss']) for h in hist if 'loss' in h]
eval_pts = [(h['step'], h['eval_loss']) for h in hist if 'eval_loss' in h]

print('== TRAIN LOSS ==')
for s, l in train_pts:
    print(f'step {s:5d} | loss {l:.4f}')
print('\n== VAL LOSS ==')
for s, l in eval_pts:
    print(f'step {s:5d} | eval_loss {l:.4f}')

print('\nepochs:', args.num_train_epochs)
if eval_pts:
    print('val_loss finale:', round(eval_pts[-1][1], 4))

# Sauvegarde des metriques dans un fichier a joindre au rendu
with open('metrics.json', 'w') as f:
    json.dump({'train_loss': train_pts, 'eval_loss': eval_pts,
               'epochs': args.num_train_epochs}, f, indent=2)


### Courbe des pertes (optionnel)


In [ ]:
import matplotlib.pyplot as plt
if train_pts:
    plt.plot(*zip(*train_pts), label='train loss')
if eval_pts:
    plt.plot(*zip(*eval_pts), label='val loss', marker='o')
plt.xlabel('step'); plt.ylabel('loss'); plt.legend(); plt.title('Fine-tuning medical Phi-3.5')
plt.show()


## 9. Test rapide du modele


In [ ]:
def ask(question, max_new_tokens=200):
    prompt = f'<|user|>\n{question}<|end|>\n<|assistant|>\n'
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    model.eval()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                             do_sample=True, temperature=0.7, top_p=0.9,
                             pad_token_id=tokenizer.eos_token_id)
    text = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return text.strip()

print(ask('I have had a sore throat and mild fever for 3 days. What should I do?'))


## 10. Sauvegarder l'adapter LoRA et le telecharger
Produit un `.zip` de l'adapter (poids LoRA) + les metriques.


In [ ]:
model.save_pretrained('phi35_medical_lora')
tokenizer.save_pretrained('phi35_medical_lora')

!cp metrics.json phi35_medical_lora/ 2>/dev/null
!zip -r phi35_medical_lora.zip phi35_medical_lora > /dev/null

from google.colab import files
files.download('phi35_medical_lora.zip')
print('Adapter + metriques telecharges (phi35_medical_lora.zip).')
